# 01 — Exploración del dataset maestro 1m

Valida descarga, timestamps (us vs ms), OHLC, gaps, volumen. Requiere `data/processed/btcusdt_1m.parquet` (generado por `cleaner.py`).

In [ ]:
import polars as pl, duckdb
from pathlib import Path
p = Path("../data/processed/btcusdt_1m.parquet")
print("exists:", p.exists())
if p.exists():
    df = pl.read_parquet(p)
    print(df.shape)
    print(df.head(3))
    print(df.describe())
    # gaps
    gaps = df.select((pl.col("timestamp").diff().dt.total_seconds()/60).alias("delta_m")).filter(pl.col("delta_m")>1.5)
    print("gaps >1m:", len(gaps))
    display(gaps.head())
else:
    print("Ejecuta primero: python -m src.ingestion.download && python -m src.processing.cleaner")


In [ ]:
# OHLC sanity + volumen
if 'df' in locals():
    invalid = df.filter((pl.col("high")<pl.col("low")) | (pl.col("high")<pl.col("open")))
    print("OHLC invalid:", len(invalid))
    print(df.select(["volume","quote_volume","trade_count","buy_ratio","volume_delta"]).describe())
